# OR Intelligence · 02 · Detection model (Member B)

Trains, evaluates and saves the instrument detector. Needs the dataset from **01_Data** (Member A).

| Step | What happens | WBS |
|---|---|---|
| 1 | Settings (the only cell you edit) | — |
| 2 | GPU check, install, connect Google Drive | — |
| 3 | Load Member A's dataset from Drive | — |
| 4 | Train the baseline model | 3.1.1 |
| 5 | Optional: one experiment | 3.1.2 |
| 6 | Pick the best model (validation split only) | 3.1.2 |
| 7 | Test-set evaluation (**runs once**) | 3.2.1 |
| 8 | Save the model to Drive | 3.3.2 |
| 9 | Run the model on a video → `detections.jsonl` | 3.3.1 |

**How to use:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Training saves straight to Drive with a checkpoint every epoch, so a disconnect does not lose work.

> Never commit `.pt` weights to GitHub. They live in `MyDrive/or-intelligence/weights/`.

## 1 · Settings — the only cell you need to edit

In [ ]:
DRIVE_DIR   = "/content/drive/MyDrive/or-intelligence"
DATASET_ZIP = f"{DRIVE_DIR}/data/yolo_dataset.zip"   # produced by 01_Data.ipynb (Member A)

MODEL    = "yolov8s.pt"   # COCO-pretrained start point (yolov8n.pt = faster, yolov8m.pt = stronger)
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16
PATIENCE = 10
SEED     = 42
CONF     = 0.25           # confidence threshold for the test report and video inference

## 2 · GPU, install, Google Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
!pip -q install ultralytics

from google.colab import drive
drive.mount("/content/drive")

import os, json, random, re, shutil, zipfile, glob, hashlib, datetime
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

for sub in ["data", "runs", "weights", "results"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
WORK = Path("/content/work")      # fast local disk; Drive keeps the results
WORK.mkdir(exist_ok=True)
print("Drive folder ready:", DRIVE_DIR)

## 3 · Load the dataset from Drive

In [ ]:
DS = WORK / "yolo"
if not DS.exists():
    assert os.path.exists(DATASET_ZIP), "Run 01_Data.ipynb first (Member A), or ask for yolo_dataset.zip"
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(DS)
yaml_text = (DS / "dataset.yaml").read_text()
(DS / "dataset.yaml").write_text(re.sub(r"^path:.*$", lambda _: f"path: {DS}", yaml_text, flags=re.M))
CLASSES = [line.split(":", 1)[1].strip() for line in yaml_text.splitlines() if re.match(r"\s+\d+:", line)]
print("classes:", CLASSES)
print({k: len(list((DS / "images" / k).iterdir())) for k in ["train", "val", "test"]}, "images per split")

## 4 · Train the baseline (WBS 3.1.1)

After it finishes, write the val mAP@0.5 into `docs/model/experiments.md`. If Colab disconnects, re-run steps 1–3 and then the **resume** cell.

In [ ]:
from ultralytics import YOLO
import pandas as pd

def train(run_name, **overrides):
    start_from = overrides.pop("model", MODEL)
    args = dict(data=str(DS / "dataset.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
                seed=SEED, deterministic=True, project=f"{DRIVE_DIR}/runs", name=run_name, exist_ok=True,
                save_period=1)
    args.update(overrides)
    YOLO(start_from).train(**args)
    return f"{DRIVE_DIR}/runs/{run_name}/weights/best.pt"

baseline_weights = train("baseline")
print("baseline weights:", baseline_weights)

In [ ]:
# RESUME after a disconnect (only if training stopped early). Uncomment and run:
# YOLO(f"{DRIVE_DIR}/runs/baseline/weights/last.pt").train(resume=True)

## 5 · Optional: one experiment (WBS 3.1.2)

Change **one** thing, chosen from Member A's EDA (for example a larger image size if tools are small). Skip it if short on time.

In [ ]:
RUN_EXPERIMENT = False      # set True to run it
if RUN_EXPERIMENT:
    exp_weights = train("exp01_imgsz832", imgsz=832)

## 6 · Pick the best model — validation split only (WBS 3.1.2)

The test split is **not** used for choosing.

In [ ]:
def val_scores(weights):
    m = YOLO(weights).val(data=str(DS / "dataset.yaml"), split="val", imgsz=IMGSZ, verbose=False)
    return {"map50": float(m.box.map50), "map50_95": float(m.box.map),
            "per_class_ap50": {CLASSES[i]: float(m.box.ap50[j]) for j, i in enumerate(m.box.ap_class_index)}}

candidates = {name: f"{DRIVE_DIR}/runs/{name}/weights/best.pt" for name in ["baseline", "exp01_imgsz832"]
              if os.path.exists(f"{DRIVE_DIR}/runs/{name}/weights/best.pt")}
scores = {name: val_scores(w) for name, w in candidates.items()}
display(pd.DataFrame({n: {"val mAP@0.5": s["map50"], "val mAP@0.5:0.95": s["map50_95"]} for n, s in scores.items()}).T)
chosen = max(scores, key=lambda n: scores[n]["map50"])
BEST = candidates[chosen]
json.dump({"chosen": chosen, "scores": scores}, open(f"{DRIVE_DIR}/results/model_selection.json", "w"), indent=2)
print("chosen model:", chosen)

## 7 · Test-set evaluation — runs ONCE (WBS 3.2.1)

mAP@0.5 and precision/recall per instrument on surgeries the model never saw. Copy the result into `docs/model/test_metrics.json` and `docs/model/EVALUATION.md`, and tell Member C the `CONF` value used.

In [ ]:
TEST_FILE = f"{DRIVE_DIR}/results/test_metrics.json"
if os.path.exists(TEST_FILE):
    print("Test split already evaluated once - showing the saved result instead of re-running.")
    test = json.load(open(TEST_FILE))
else:
    m = YOLO(BEST).val(data=str(DS / "dataset.yaml"), split="test", imgsz=IMGSZ, conf=CONF, plots=True,
                       project=f"{DRIVE_DIR}/runs", name="test_eval", exist_ok=True)
    per_class = {}
    for j, i in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(j)
        per_class[CLASSES[i]] = {"precision": float(p), "recall": float(r), "ap50": float(ap50)}
    test = {"model": chosen, "weights_sha256": hashlib.sha256(open(BEST, "rb").read()).hexdigest(),
            "conf": CONF, "map50": float(m.box.map50), "map50_95": float(m.box.map), "per_class": per_class,
            "evaluated_at": datetime.datetime.now().isoformat(timespec="seconds")}
    json.dump(test, open(TEST_FILE, "w"), indent=2)
display(pd.DataFrame(test["per_class"]).T.round(3))
print(f"TEST mAP@0.5 = {test['map50']:.3f}   mAP@0.5:0.95 = {test['map50_95']:.3f}")

## 8 · Save the final model (WBS 3.3.2)

`best.pt` (+ `best.onnx`, + sha256) go to `MyDrive/or-intelligence/weights/`. Share that folder with the team, or attach `best.pt` to a GitHub Release `model-v1`. Then write `docs/model/MODEL_CARD.md`.

In [ ]:
final_pt = f"{DRIVE_DIR}/weights/best.pt"
shutil.copy(BEST, final_pt)
onnx_path = YOLO(final_pt).export(format="onnx", imgsz=IMGSZ)
shutil.copy(onnx_path, f"{DRIVE_DIR}/weights/best.onnx")
sha = hashlib.sha256(open(final_pt, "rb").read()).hexdigest()
open(f"{DRIVE_DIR}/weights/best.pt.sha256", "w").write(sha)
print("saved:", final_pt, "\nsha256:", sha)
# from google.colab import files; files.download(final_pt)     # optional: download to this computer

## 9 · Run the model on a video → `detections.jsonl` (WBS 3.3.1)

This is the same function as `src/inference/detect_video.py` in the repo (which the app uses). Output follows contract C4 (`contracts/detections.schema.json`). Upload a video to Drive and set `VIDEO`.

In [ ]:
import cv2

def detect_video(video_path, out_path, sample_fps=5.0, conf=CONF, weights=final_pt, on_progress=None):
    model = YOLO(weights)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    step = max(1, round(fps / sample_fps))
    vid, written, idx = Path(video_path).stem, 0, 0
    with open(out_path, "w") as f:
        while cap.grab():
            if idx % step == 0:
                ok, frame = cap.retrieve()
                if not ok:
                    break
                r = model.predict(frame, conf=conf, imgsz=IMGSZ, verbose=False)[0]
                dets = [{"cls": int(c), "name": CLASSES[int(c)], "conf": round(float(s), 4),
                         "box": [round(min(max(float(v), 0.0), 1.0), 4) for v in b]}
                        for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxyn)]
                f.write(json.dumps({"video_id": vid, "frame": idx, "t": round(idx / fps, 3), "w": W, "h": H, "dets": dets}) + "\n")
                written += 1
                if on_progress and total:
                    on_progress(min(idx / total, 1.0))
            idx += 1
    cap.release()
    if on_progress:
        on_progress(1.0)
    return {"frames": written, "seconds": (total or idx) / fps}

VIDEO = f"{DRIVE_DIR}/data/sample_video.mp4"     # upload any test video here
if os.path.exists(VIDEO):
    out = f"{DRIVE_DIR}/results/{Path(VIDEO).stem}_detections.jsonl"
    print(detect_video(VIDEO, out), "->", out)
else:
    print("No video at", VIDEO, "- skip this step or upload one.")

## Done — hand-off

| File in Drive | Who needs it | Repo location |
|---|---|---|
| `weights/best.pt` (+ `.onnx`, `.sha256`) | Member C (API, timeline eval), Member D (demo) | Drive / GitHub Release `model-v1`, **never committed** |
| `results/test_metrics.json` | Member D (release), report, slides | `docs/model/test_metrics.json` |
| `results/model_selection.json` | Report | numbers into `docs/model/experiments.md` |
| `runs/test_eval/*.png` | Report, slides | `docs/model/figures/` |
| `results/*_detections.jsonl` | Member C (timeline) | input to `src/timeline/builder.py` |

Only write numbers that came out of this notebook.